<a href="https://colab.research.google.com/github/carlariqc-glitch/Modelo-google-colab-gato-VS-perro/blob/main/perro_gato_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fase 1: **Cargar modelo**


In [15]:
# 1. Activamos el modo legacy (Keras 2)
import os

os.environ['TF_USE_LEGACY_KERAS'] = '1'
from PIL import Image, ImageOps
import numpy as np
from tensorflow import keras

#/content/drive/MyDrive/converted_keras (Unzipped Files)
mi_modelo = keras.models.load_model("/content/drive/MyDrive/modelo/keras_model.h5", compile=False)

# Carga las etiquetas de las clases
nombre_clases = open("/content/drive/MyDrive/modelo/labels.txt", "r").readlines()

print(nombre_clases)



['0 gato\n', '1 perro\n']


In [16]:
lista_archivos = os.listdir("/content/drive/MyDrive/perros y gatos/imagenes/test")


In [17]:
total_predicciones = 0
aciertos = 0
media_probabilidad_aciertos = 0.0
# predicciones incorrectas contendrá una lista con los nombres de las imágenes mal clasificadas
predicciones_incorrectas = []

In [18]:
def predecir_imagen(rutaimg):
    imagen = Image.open(rutaimg).convert("RGB")
    #preprocesamiento
    size = (224, 224)
    imagen = ImageOps.fit(imagen, size, Image.Resampling.LANCZOS)
    # Convertimos la imagen en un array NumPy.
    imagen_array = np.asarray(imagen)
    normalizada_imagen_array = (imagen_array.astype(np.float32) / 127.5) - 1
    # Crear un array para un lote de 1 imagen. ndarray = N-Dimensional Array
    lote_imagenes = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
    lote_imagenes[0] = normalizada_imagen_array
    resultados= mi_modelo.predict(lote_imagenes)
    indice = np.argmax(resultados[0])
    #postprocesamiento
    eti = nombre_clases[indice]

    prob = resultados[0][indice]
    return eti, prob


In [19]:
for nombre_archivo in lista_archivos:
    if "cat" in nombre_archivo:
        etiqueta_esperada = "gato"
    elif "dog" in nombre_archivo:
        etiqueta_esperada = "perro"
    else:
        continue  # Saltar archivos que no sean de gatos o perros

    total_predicciones += 1

    ruta_imagen = os.path.join("/content/drive/MyDrive/perros y gatos/imagenes/test/", nombre_archivo)

    # Aquí obtendriamos la predicción del modelo llamando a una función que
    # implemente las fases de inferencia
    etiqueta_predicha, probabilidad = predecir_imagen(ruta_imagen)

    if  etiqueta_esperada in etiqueta_predicha:
        aciertos += 1
        media_probabilidad_aciertos += probabilidad
    else:
        info_error= {"archivo": nombre_archivo, "prediccion": etiqueta_predicha,
             "probabilidad": probabilidad}
        predicciones_incorrectas.append(info_error)


1/1 [==============================] - 0s 22ms/step


In [11]:
precision = aciertos / total_predicciones if total_predicciones > 0 else 0
media_probabilidad_aciertos /= aciertos if aciertos > 0 else 1



In [20]:
informe = f"""
Informe de Evaluación del Modelo

Total de predicciones: {total_predicciones}
Aciertos: {aciertos}
Precisión: {precision:.2%}
Probabilidad media de aciertos: {media_probabilidad_aciertos:.2%}

Predicciones incorrectas:
"""

print (informe)



Informe de Evaluación del Modelo

Total de predicciones: 400
Aciertos: 391
Precisión: 48.75%
Probabilidad media de aciertos: 38774.20%

Predicciones incorrectas:



In [21]:
for error in predicciones_incorrectas:
    informe += f"Archivo: {error['archivo']}, Predicción: {error['prediccion']}, Probabilidad: {error['probabilidad']:.2%}\n"

print(informe)


Informe de Evaluación del Modelo

Total de predicciones: 400
Aciertos: 391
Precisión: 48.75%
Probabilidad media de aciertos: 38774.20%

Predicciones incorrectas:
Archivo: cat.9920.jpg, Predicción: 1 perro
, Probabilidad: 70.21%
Archivo: cat.9882.jpg, Predicción: 1 perro
, Probabilidad: 94.77%
Archivo: cat.9960.jpg, Predicción: 1 perro
, Probabilidad: 99.92%
Archivo: cat.9947.jpg, Predicción: 1 perro
, Probabilidad: 77.11%
Archivo: cat.9974.jpg, Predicción: 1 perro
, Probabilidad: 74.11%
Archivo: dog.9935.jpg, Predicción: 0 gato
, Probabilidad: 98.92%
Archivo: dog.9911.jpg, Predicción: 0 gato
, Probabilidad: 99.92%
Archivo: dog.9868.jpg, Predicción: 0 gato
, Probabilidad: 85.26%
Archivo: dog.9808.jpg, Predicción: 0 gato
, Probabilidad: 76.87%

